In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Helper Classes

In [ ]:
class Comparison:
    def __init__(self, str_uri, str_dirname_output):
        self.str_uri = str_uri
        self.str_dirname_output = str_dirname_output
        self.df_data = None
        self.df_freq = None
        self.df_bayes = None

    def import_data(self):
        self.df_data = pd.read_parquet(self.str_uri)
        self.df_freq = pd.read_csv('../03_frequentist_test/output/frequentist_results.csv')
        self.df_bayes = pd.read_csv('../04_bayesian_test/output/bayesian_results.csv')
        print(f'Data loaded: {self.df_data.shape[0]:,} rows')
        print(f'Frequentist results: {len(self.df_freq)} tests')
        print(f'Bayesian results: {len(self.df_bayes)} tests')
        return self

    def plot_methodology_comparison(self):
        df_freq_retention = self.df_freq[self.df_freq['metric'].isin(['1-Day', '7-Day'])].copy()
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        list_metrics = df_freq_retention['metric'].tolist()
        # Frequentist CIs
        for i, (_, row) in enumerate(df_freq_retention.iterrows()):
            flt_diff = row['difference'] * 100
            flt_ci_l = row['ci_lower'] * 100
            flt_ci_u = row['ci_upper'] * 100
            axes[0].errorbar(flt_diff, i, xerr=[[flt_diff - flt_ci_l], [flt_ci_u - flt_diff]],
                             fmt='o', color='#4C72B0', markersize=10, capsize=8, capthick=2, linewidth=2)
            axes[0].annotate(f'{flt_diff:.2f}pp [{flt_ci_l:.2f}, {flt_ci_u:.2f}]',
                             xy=(flt_diff, i), xytext=(0, 15), textcoords='offset points',
                             ha='center', fontsize=10)
        axes[0].axvline(x=0, color='red', linestyle='--')
        axes[0].set_yticks(range(len(list_metrics)))
        axes[0].set_yticklabels(list_metrics)
        axes[0].set_xlabel('Difference (Percentage Points)')
        axes[0].set_title('Frequentist 95% CI', fontsize=14, y=1.02)
        axes[0].grid(True, alpha=0.3, axis='x')
        # Bayesian HDIs
        for i, (_, row) in enumerate(self.df_bayes.iterrows()):
            flt_mean = (row['hdi_lower'] + row['hdi_upper']) / 2 * 100
            flt_hdi_l = row['hdi_lower'] * 100
            flt_hdi_u = row['hdi_upper'] * 100
            axes[1].errorbar(flt_mean, i, xerr=[[flt_mean - flt_hdi_l], [flt_hdi_u - flt_mean]],
                             fmt='s', color='#DD8452', markersize=10, capsize=8, capthick=2, linewidth=2)
            axes[1].annotate(f'{flt_mean:.2f}pp [{flt_hdi_l:.2f}, {flt_hdi_u:.2f}]',
                             xy=(flt_mean, i), xytext=(0, 15), textcoords='offset points',
                             ha='center', fontsize=10)
        axes[1].axvline(x=0, color='red', linestyle='--')
        axes[1].set_yticks(range(len(self.df_bayes)))
        axes[1].set_yticklabels(self.df_bayes['metric'].tolist())
        axes[1].set_xlabel('Difference (Percentage Points)')
        axes[1].set_title('Bayesian 95% HDI', fontsize=14, y=1.02)
        axes[1].grid(True, alpha=0.3, axis='x')
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/01_methodology_comparison.png', bbox_inches='tight', dpi=150)
        plt.show()
        return self

    def plot_overall_retention_summary(self):
        df_summary = self.df_data.groupby('version').agg(
            flt_retention_1=('retention_1', 'mean'),
            flt_retention_7=('retention_7', 'mean'),
            flt_mean_rounds=('sum_gamerounds', 'mean')
        ).reset_index()
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        list_labels = ['gate_30', 'gate_40']
        list_colors = ['#4C72B0', '#DD8452']
        # 1-day retention
        list_vals = df_summary['flt_retention_1'].values * 100
        bars = axes[0].bar(list_labels, list_vals, color=list_colors, edgecolor='black')
        axes[0].set_title('1-Day Retention', fontsize=14, y=1.02)
        axes[0].set_ylabel('Retention Rate (%)')
        for bar, flt_v in zip(bars, list_vals):
            axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
                         f'{flt_v:.2f}%', ha='center', fontsize=11)
        # 7-day retention
        list_vals = df_summary['flt_retention_7'].values * 100
        bars = axes[1].bar(list_labels, list_vals, color=list_colors, edgecolor='black')
        axes[1].set_title('7-Day Retention', fontsize=14, y=1.02)
        axes[1].set_ylabel('Retention Rate (%)')
        for bar, flt_v in zip(bars, list_vals):
            axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
                         f'{flt_v:.2f}%', ha='center', fontsize=11)
        # Mean game rounds
        list_vals = df_summary['flt_mean_rounds'].values
        bars = axes[2].bar(list_labels, list_vals, color=list_colors, edgecolor='black')
        axes[2].set_title('Mean Game Rounds', fontsize=14, y=1.02)
        axes[2].set_ylabel('Average Rounds Played')
        for bar, flt_v in zip(bars, list_vals):
            axes[2].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                         f'{flt_v:.1f}', ha='center', fontsize=11)
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/02_overall_summary.png', bbox_inches='tight', dpi=150)
        plt.show()
        return self

    def print_combined_results(self):
        print('\n=== COMBINED RESULTS ===')
        print('\n--- Frequentist Results ---')
        print(self.df_freq.to_string(index=False))
        print('\n--- Bayesian Results ---')
        print(self.df_bayes.to_string(index=False))
        print('\n--- Group Summary ---')
        df_summary = self.df_data.groupby('version').agg(
            n_players=('userid', 'count'),
            retention_1=('retention_1', 'mean'),
            retention_7=('retention_7', 'mean'),
            mean_gamerounds=('sum_gamerounds', 'mean'),
            median_gamerounds=('sum_gamerounds', 'median')
        ).reset_index()
        df_summary['retention_1'] = df_summary['retention_1'].map('{:.4f}'.format)
        df_summary['retention_7'] = df_summary['retention_7'].map('{:.4f}'.format)
        df_summary['mean_gamerounds'] = df_summary['mean_gamerounds'].map('{:.2f}'.format)
        print(df_summary.to_string(index=False))
        return self

    def save_combined_results(self):
        df_combined = pd.DataFrame([
            {'aspect': 'Approach', 'frequentist': 'Hypothesis Testing', 'bayesian': 'Posterior Probability'},
            {'aspect': 'Key Output', 'frequentist': 'P-value + CI', 'bayesian': 'P(A > B) + HDI'},
        ])
        for _, row in self.df_freq[self.df_freq['metric'].isin(['1-Day', '7-Day'])].iterrows():
            str_m = row['metric']
            df_combined = pd.concat([df_combined, pd.DataFrame([{
                'aspect': f'{str_m} Significant?',
                'frequentist': f'p={row["p_value"]:.4f} ({"Yes" if row["significant"] else "No"})',
                'bayesian': ''
            }])], ignore_index=True)
        for _, row in self.df_bayes.iterrows():
            str_m = row['metric']
            idx = df_combined[df_combined['aspect'] == f'{str_m} Significant?'].index
            if len(idx) > 0:
                df_combined.loc[idx[0], 'bayesian'] = f'P(Control better)={row["p_control_better"]:.2%}'
        df_combined.to_csv(f'{self.str_dirname_output}/combined_results.csv', index=False)
        print(f'\nCombined results saved to {self.str_dirname_output}/combined_results.csv')
        print(df_combined.to_string(index=False))
        return self

## Constants

In [ ]:
str_bucket = 'ab-testing-demo-repo'
str_task = 'comparison'
str_dirname_output = './output'
str_uri = f's3://{str_bucket}/00_data_collection/cookie_cats.parquet'

## Output Directory

In [ ]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass
print(f'Output directory ready: {str_dirname_output}')

## Run Comparison

In [ ]:
cls_comp = Comparison(str_uri, str_dirname_output)
cls_comp.import_data()

In [ ]:
cls_comp.plot_methodology_comparison()

In [ ]:
cls_comp.plot_overall_retention_summary()

In [ ]:
cls_comp.print_combined_results()

In [ ]:
cls_comp.save_combined_results()

## Completion

In [ ]:
print('\n=== COMPARISON COMPLETE ===')
print(f'All results saved to: {str_dirname_output}')